In [1]:
# 라이브러리 실행 확인용
import pandas as pd
import numpy as np

In [1]:
#(role_performance_distribution_kr.csv) 팀포지션별 승패별 평균 골드,딜량,시야,팀내 비중 
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# 1. 파일 경로 설정 및 데이터 불러오기
file_path = r"D:\바탕화면\데이터 분석\프로젝트\04 최종 프로젝트\구글 드라이브\details\전처리 후\패치 버전 15미만 제외\details_kr_part1_flattened_v15_16.csv"
df = pd.read_csv(file_path)

# 2. 필수 데이터 결측치 제거
df_valid = df.dropna(subset=['teamPosition', 'win', 'gameDuration', 'endOfGameResult']).copy()

# 3. 서렌 가능 시간인 게임 시작 후 15분(900초) 이상 진행된 매치만
cond_duration = df_valid['gameDuration'] >= 900

# 4. 끝까지 정상적으로 완료된 매치(GameComplete)만
cond_result = df_valid['endOfGameResult'] == 'GameComplete'

# 두 조건을 모두 만족하는 데이터만 덮어쓰기
df_filtered = df_valid[cond_duration & cond_result].copy()
df_filtered['win'] = df_filtered['win'].astype(int)

print(f"필터링 전 데이터 수: {len(df_valid)}행")
print(f"필터링 후 데이터 수: {len(df_filtered)}행\n")

# 5. 분석할 주요 지표 설정 (골드, 딜량, 시야, 팀내 딜 비중)
metrics = ['goldEarned', 'totalDamageDealtToChampions', 'visionScore']
if 'ch_teamDamagePercentage' in df_filtered.columns:
    metrics.append('ch_teamDamagePercentage')

# 6. 승패 및 포지션별 집계 (평균, 중앙값, 표준편차)
agg_funcs = ['mean', 'median', 'std']
grouped_stats = df_filtered.groupby(['teamPosition', 'win'])[metrics].agg(agg_funcs)

# 멀티인덱스로 묶인 컬럼명을 하나로 평탄화
grouped_stats.columns = [f"{col}_{func}" for col, func in grouped_stats.columns]
grouped_stats = grouped_stats.reset_index()

# 7. 보기 좋게 데이터 정제 (한글화 및 순서 정렬)
grouped_stats['win'] = grouped_stats['win'].map({1: '승리', 0: '패배'})

# 포지션 정렬 (탑 -> 정글 -> 미드 -> 원딜 -> 유틸)
desired_order = ['TOP', 'JUNGLE', 'MIDDLE', 'BOTTOM', 'UTILITY']
grouped_stats['teamPosition'] = pd.Categorical(grouped_stats['teamPosition'], categories=desired_order, ordered=True)
grouped_stats = grouped_stats.sort_values(by=['teamPosition', 'win'], ascending=[True, False])

# 컬럼명 직관적으로 변경하는 함수
def rename_columns(col_name):
    name = col_name
    name = name.replace('goldEarned', '골드').replace('totalDamageDealtToChampions', '딜량')
    name = name.replace('visionScore', '시야').replace('ch_teamDamagePercentage', '팀내_딜비중(%)')
    name = name.replace('_mean', '_평균').replace('_median', '_중앙값').replace('_std', '_표준편차')
    name = name.replace('teamPosition', '포지션').replace('win', '승패')
    return name

grouped_stats.rename(columns=lambda x: rename_columns(x), inplace=True)

# 소수점 반올림 (가독성을 위해)
numeric_cols = grouped_stats.select_dtypes(include=['float64', 'int64']).columns
grouped_stats[numeric_cols] = grouped_stats[numeric_cols].round(2)

# 8. 결과 확인
print("조건이 적용된 포지션 및 승패별 분포/구조/성과 지표:")
display(grouped_stats)

# 9. 결과를 CSV 파일로 저장
output_path = r"D:\바탕화면\데이터 분석\프로젝트\04 최종 프로젝트\구글 드라이브\details\전처리 후\패치 버전 15미만 제외\role_performance_distribution_kr.csv"
grouped_stats.to_csv(output_path, index=False, encoding='utf-8-sig')

print(f"\n데이터 처리가 완료되었으며, 저장 위치는 다음과 같습니다:\n{output_path}")

필터링 전 데이터 수: 173959행
필터링 후 데이터 수: 168919행

조건이 적용된 포지션 및 승패별 분포/구조/성과 지표:


,포지션,승패,골드_평균,골드_중앙값,골드_표준편차,딜량_평균,딜량_중앙값,딜량_표준편차,시야_평균,시야_중앙값,시야_표준편차,팀내_딜비중(%)_평균,팀내_딜비중(%)_중앙값,팀내_딜비중(%)_표준편차
6,TOP,패배,10749.86,10541.5,3733.26,21691.56,19534.5,11981.89,17.67,16.0,9.65,0.23,0.22,0.07
7,TOP,승리,12553.81,12468.0,3621.76,25826.54,23981.0,12642.45,18.83,17.0,9.83,0.23,0.22,0.07
2,JUNGLE,패배,11219.93,10993.0,3652.34,17870.50,15849.5,10804.72,25.70,24.0,12.18,0.18,0.18,0.06
3,JUNGLE,승리,13093.40,13084.0,3696.23,21593.91,19773.0,11867.02,27.94,26.0,12.40,0.18,0.18,0.06
4,MIDDLE,패배,10874.82,10699.0,3652.32,21715.08,19783.0,11977.16,19.24,18.0,10.11,0.23,0.22,0.06
5,MIDDLE,승리,12486.20,12411.0,3523.81,25958.58,24283.0,12583.34,20.91,20.0,10.16,0.23,0.22,0.07
0,BOTTOM,패배,11769.77,11500.0,4152.69,21669.64,19097.0,13145.25,17.27,16.0,9.35,0.22,0.22,0.07
1,BOTTOM,승리,13848.60,13722.0,4103.68,27424.08,25047.0,14410.20,18.67,17.0,9.58,0.24,0.23,0.08
8,UTILITY,패배,8144.07,7942.0,2600.45,12788.86,10446.5,8761.82,62.04,60.0,29.85,0.13,0.12,0.06
9,UTILITY,승리,9271.13,9185.0,2539.29,14175.40,11774.0,9098.40,66.70,65.0,29.00,0.12,0.11,0.06



데이터 처리가 완료되었으며, 저장 위치는 다음과 같습니다:
D:\바탕화면\데이터 분석\프로젝트\04 최종 프로젝트\구글 드라이브\details\전처리 후\패치 버전 15미만 제외\role_performance_distribution_kr.csv


In [5]:
#(champion_comprehensive_stats_kr.csv)챔피언별 팀라인별 매치수 픽률/승률/평균(골드/딜/시야점수)
import pandas as pd
import numpy as np

# 1. 파일 경로 설정 및 데이터 불러오기
file_path = r"D:\바탕화면\데이터 분석\프로젝트\04 최종 프로젝트\구글 드라이브\details\전처리 후\패치 버전 15미만 제외\details_kr_part1_flattened_v15_16.csv"
df = pd.read_csv(file_path)

# 2. 필수 데이터 결측치 제거
df_valid = df.dropna(subset=['championName', 'teamPosition', 'win', 'gameDuration', 'endOfGameResult']).copy()

# 3. 필터링 조건 적용
# 조건 1: 서렌 가능 시간인 게임 시작 후 15분(900초) 이상 진행된 매치만
cond_duration = df_valid['gameDuration'] >= 900

# 조건 2: 끝까지 정상적으로 완료된 매치(GameComplete)만
cond_result = df_valid['endOfGameResult'] == 'GameComplete'

# 조건을 모두 만족하는 정상 데이터만 추출
df_filtered = df_valid[cond_duration & cond_result].copy()
df_filtered['win'] = df_filtered['win'].astype(int)

print(f"필터링 전 전체 데이터 수: {len(df_valid)}행")
print(f"조건 적용 후 데이터 수: {len(df_filtered)}행\n")

# 4. [기본 지표 계산] 매치 수, 픽률, 승률
match_count = pd.crosstab(index=df_filtered['championName'], columns=df_filtered['teamPosition'])
pick_rate = pd.crosstab(index=df_filtered['championName'], columns=df_filtered['teamPosition'], normalize='index') * 100
win_rate = pd.pivot_table(data=df_filtered, values='win', index='championName', columns='teamPosition', aggfunc='mean') * 100

# 5. [심화 지표 계산] 골드, 딜량, 시야 점수 평균
metrics = ['goldEarned', 'totalDamageDealtToChampions', 'visionScore']
grouped_avg = df_filtered.groupby(['championName', 'teamPosition'])[metrics].mean()
pivoted_avg = grouped_avg.unstack(level='teamPosition')

# 6. 열 순서 지정 및 결측치 처리
desired_order = ['TOP', 'JUNGLE', 'MIDDLE', 'BOTTOM', 'UTILITY']

match_count = match_count.reindex(columns=desired_order).fillna(0)
pick_rate = pick_rate.reindex(columns=desired_order).fillna(0)
win_rate = win_rate.reindex(columns=desired_order).fillna(0)

# 7. 모든 지표를 하나의 데이터프레임으로 병합
result_df = pd.DataFrame(index=pick_rate.index)

for pos in desired_order:
    # 7-1. 기본 지표 추가
    result_df[f'{pos}_매치수'] = match_count[pos].astype(int)
    result_df[f'{pos}_픽률(%)'] = pick_rate[pos].round(2)
    result_df[f'{pos}_승률(%)'] = win_rate[pos].round(2)
    
    # 7-2. 심화 지표(골드, 딜, 시야) 추가
    if ('goldEarned', pos) in pivoted_avg.columns:
        result_df[f'{pos}_골드평균'] = pivoted_avg[('goldEarned', pos)].fillna(0).round(1)
        result_df[f'{pos}_딜평균'] = pivoted_avg[('totalDamageDealtToChampions', pos)].fillna(0).round(1)
        result_df[f'{pos}_시야평균'] = pivoted_avg[('visionScore', pos)].fillna(0).round(1)
    else:
        result_df[f'{pos}_골드평균'] = 0.0
        result_df[f'{pos}_딜평균'] = 0.0
        result_df[f'{pos}_시야평균'] = 0.0

# 8. 결과 확인 (상위 15개 챔피언)
print("챔피언(행) x 포지션별 종합 지표 (조건 적용 완료):")
display(result_df.head(15))

# 9. 결과를 CSV 파일로 저장
output_path = r"D:\바탕화면\데이터 분석\프로젝트\04 최종 프로젝트\구글 드라이브\details\전처리 후\패치 버전 15미만 제외\champion_comprehensive_stats_kr.csv"
result_df.to_csv(output_path, encoding='utf-8-sig')

print(f"\n정상 조건이 적용된 통합 마스터 데이터가 성공적으로 저장되었습니다:\n{output_path}")

필터링 전 전체 데이터 수: 173959행
조건 적용 후 데이터 수: 168919행

챔피언(행) x 포지션별 종합 지표 (조건 적용 완료):


,TOP_매치수,TOP_픽률(%),TOP_승률(%),TOP_골드평균,TOP_딜평균,TOP_시야평균,JUNGLE_매치수,JUNGLE_픽률(%),JUNGLE_승률(%),JUNGLE_골드평균,...,BOTTOM_승률(%),BOTTOM_골드평균,BOTTOM_딜평균,BOTTOM_시야평균,UTILITY_매치수,UTILITY_픽률(%),UTILITY_승률(%),UTILITY_골드평균,UTILITY_딜평균,UTILITY_시야평균
championName,,,,,,,,,,,,,,,,,,,,,
Aatrox,1372,81.72,48.69,11203.5,23538.1,17.8,240,14.29,48.75,11889.7,...,100.00,8050.0,13482.0,8.5,11,0.66,27.27,8342.3,14749.1,44.3
Ahri,20,1.20,60.00,12563.6,27640.2,25.6,0,0.00,0.00,0.0,...,50.00,11091.0,17072.5,25.5,38,2.27,23.68,8786.4,16695.4,54.7
Akali,385,17.41,51.69,11702.8,26015.5,20.9,0,0.00,0.00,0.0,...,0.00,11548.2,21125.2,23.2,4,0.18,25.00,10275.5,21716.8,45.0
Akshan,11,4.45,45.45,14114.8,27836.4,23.1,0,0.00,0.00,0.0,...,12.50,9915.9,17909.5,13.1,6,2.43,50.00,9879.3,17615.0,38.2
Alistar,20,2.30,30.00,9067.0,16714.6,16.4,3,0.35,33.33,9475.3,...,0.00,11895.3,17641.7,15.0,831,95.74,50.66,7929.3,8533.4,63.6
Ambessa,934,79.76,47.86,11201.1,24313.1,18.0,171,14.60,50.88,12136.0,...,0.00,10064.2,19227.2,14.2,2,0.17,100.00,11450.5,28465.5,35.5
Amumu,3,0.53,33.33,9952.0,17724.3,13.7,445,78.90,55.28,11075.7,...,0.00,0.0,0.0,0.0,116,20.57,56.03,8685.2,12526.2,58.4
Anivia,33,7.69,45.45,10932.7,21632.5,21.5,1,0.23,0.00,3689.0,...,50.00,9641.7,11858.0,15.0,93,21.68,46.24,9264.3,15853.4,65.2
Annie,22,6.65,40.91,11184.8,26106.2,22.0,0,0.00,0.00,0.0,...,50.00,9272.8,16337.5,15.3,42,12.69,52.38,9539.4,20193.2,68.2



정상 조건이 적용된 통합 마스터 데이터가 성공적으로 저장되었습니다:
D:\바탕화면\데이터 분석\프로젝트\04 최종 프로젝트\구글 드라이브\details\전처리 후\패치 버전 15미만 제외\champion_comprehensive_stats_kr.csv


In [2]:
#(champion_comprehensive_offmeta_kr.csv)챔피언별 팀라인별 픽률 1등 제외한 파일
import pandas as pd
import numpy as np

# 1. 파일 경로 설정 및 데이터 불러오기
file_path = r"D:\바탕화면\데이터 분석\프로젝트\04 최종 프로젝트\구글 드라이브\details\전처리 후\패치 버전 15미만 제외\details_kr_part1_flattened_v15_16.csv"
df = pd.read_csv(file_path)

# 2. 필수 데이터 결측치 제거
df_valid = df.dropna(subset=['championName', 'teamPosition', 'win', 'gameDuration', 'endOfGameResult']).copy()

# 3. 필터링 조건 적용 (15분 이상 & 정상 종료)
cond_duration = df_valid['gameDuration'] >= 900
cond_result = df_valid['endOfGameResult'] == 'GameComplete'

df_filtered = df_valid[cond_duration & cond_result].copy()
df_filtered['win'] = df_filtered['win'].astype(int)

# 4. [기본 지표 계산] 전체 매치 수, 픽률, 승률
match_count = pd.crosstab(index=df_filtered['championName'], columns=df_filtered['teamPosition'])
pick_rate = pd.crosstab(index=df_filtered['championName'], columns=df_filtered['teamPosition'], normalize='index') * 100
win_rate = pd.pivot_table(data=df_filtered, values='win', index='championName', columns='teamPosition', aggfunc='mean') * 100

# 5. [심화 지표 계산] 전체 골드, 딜량, 시야 점수 평균
metrics = ['goldEarned', 'totalDamageDealtToChampions', 'visionScore']
grouped_avg = df_filtered.groupby(['championName', 'teamPosition'])[metrics].mean()
pivoted_avg = grouped_avg.unstack(level='teamPosition')

# 6. 모든 지표를 하나의 데이터프레임으로 병합
desired_order = ['TOP', 'JUNGLE', 'MIDDLE', 'BOTTOM', 'UTILITY']

match_count = match_count.reindex(columns=desired_order).fillna(0)
pick_rate = pick_rate.reindex(columns=desired_order).fillna(0)
win_rate = win_rate.reindex(columns=desired_order).fillna(0)

result_df = pd.DataFrame(index=pick_rate.index)

for pos in desired_order:
    result_df[f'{pos}_매치수'] = match_count[pos].astype(int)
    result_df[f'{pos}_픽률(%)'] = pick_rate[pos].round(2)
    result_df[f'{pos}_승률(%)'] = win_rate[pos].round(2)
    
    if ('goldEarned', pos) in pivoted_avg.columns:
        result_df[f'{pos}_골드평균'] = pivoted_avg[('goldEarned', pos)].fillna(0).round(1)
        result_df[f'{pos}_딜평균'] = pivoted_avg[('totalDamageDealtToChampions', pos)].fillna(0).round(1)
        result_df[f'{pos}_시야평균'] = pivoted_avg[('visionScore', pos)].fillna(0).round(1)
    else:
        result_df[f'{pos}_골드평균'] = 0.0
        result_df[f'{pos}_딜평균'] = 0.0
        result_df[f'{pos}_시야평균'] = 0.0

# 7. ⭐️ 챔피언별 픽률 1위(주 포지션) 찾아서 해당 데이터만 비우기 ⭐️
# 챔피언별로 매치수가 가장 많은 포지션을 찾습니다.
main_positions = df_filtered.groupby(['championName', 'teamPosition']).size().groupby(level=0).idxmax().apply(lambda x: x[1])

for champ, main_pos in main_positions.items():
    if champ in result_df.index:
        # 주 포지션에 해당하는 6개 컬럼 이름 리스트 생성
        cols_to_clear = [
            f'{main_pos}_매치수', f'{main_pos}_픽률(%)', f'{main_pos}_승률(%)',
            f'{main_pos}_골드평균', f'{main_pos}_딜평균', f'{main_pos}_시야평균'
        ]
        # 해당 셀의 데이터를 NaN(빈 값)으로 만듭니다.
        result_df.loc[champ, cols_to_clear] = np.nan

# 8. 결과 확인 (상위 15개 챔피언)
print("주 포지션(1위) 제외 완료! 부포지션 종합 지표:")
display(result_df.head(15))

# 9. 결과를 CSV 파일로 저장
output_path = r"D:\바탕화면\데이터 분석\프로젝트\04 최종 프로젝트\구글 드라이브\details\전처리 후\패치 버전 15미만 제외\champion_comprehensive_offmeta_kr.csv"
result_df.to_csv(output_path, encoding='utf-8-sig')

print(f"\n주 포지션이 제외된 부포지션 통합 데이터가 성공적으로 저장되었습니다:\n{output_path}")

주 포지션(1위) 제외 완료! 부포지션 종합 지표:


,TOP_매치수,TOP_픽률(%),TOP_승률(%),TOP_골드평균,TOP_딜평균,TOP_시야평균,JUNGLE_매치수,JUNGLE_픽률(%),JUNGLE_승률(%),JUNGLE_골드평균,...,BOTTOM_승률(%),BOTTOM_골드평균,BOTTOM_딜평균,BOTTOM_시야평균,UTILITY_매치수,UTILITY_픽률(%),UTILITY_승률(%),UTILITY_골드평균,UTILITY_딜평균,UTILITY_시야평균
championName,,,,,,,,,,,,,,,,,,,,,
Aatrox,NaN,NaN,NaN,NaN,NaN,NaN,240.0,14.29,48.75,11889.7,...,100.00,8050.0,13482.0,8.5,11.0,0.66,27.27,8342.3,14749.1,44.3
Ahri,20.0,1.20,60.00,12563.6,27640.2,25.6,0.0,0.00,0.00,0.0,...,50.00,11091.0,17072.5,25.5,38.0,2.27,23.68,8786.4,16695.4,54.7
Akali,385.0,17.41,51.69,11702.8,26015.5,20.9,0.0,0.00,0.00,0.0,...,0.00,11548.2,21125.2,23.2,4.0,0.18,25.00,10275.5,21716.8,45.0
Akshan,11.0,4.45,45.45,14114.8,27836.4,23.1,0.0,0.00,0.00,0.0,...,12.50,9915.9,17909.5,13.1,6.0,2.43,50.00,9879.3,17615.0,38.2
Alistar,20.0,2.30,30.00,9067.0,16714.6,16.4,3.0,0.35,33.33,9475.3,...,0.00,11895.3,17641.7,15.0,NaN,NaN,NaN,NaN,NaN,NaN
Ambessa,NaN,NaN,NaN,NaN,NaN,NaN,171.0,14.60,50.88,12136.0,...,0.00,10064.2,19227.2,14.2,2.0,0.17,100.00,11450.5,28465.5,35.5
Amumu,3.0,0.53,33.33,9952.0,17724.3,13.7,NaN,NaN,NaN,NaN,...,0.00,0.0,0.0,0.0,116.0,20.57,56.03,8685.2,12526.2,58.4
Anivia,33.0,7.69,45.45,10932.7,21632.5,21.5,1.0,0.23,0.00,3689.0,...,50.00,9641.7,11858.0,15.0,93.0,21.68,46.24,9264.3,15853.4,65.2
Annie,22.0,6.65,40.91,11184.8,26106.2,22.0,0.0,0.00,0.00,0.0,...,50.00,9272.8,16337.5,15.3,42.0,12.69,52.38,9539.4,20193.2,68.2



주 포지션이 제외된 부포지션 통합 데이터가 성공적으로 저장되었습니다:
D:\바탕화면\데이터 분석\프로젝트\04 최종 프로젝트\구글 드라이브\details\전처리 후\패치 버전 15미만 제외\champion_comprehensive_offmeta_kr.csv


In [4]:
#(champion_comprehensive_extended_offmeta_kr.csv)챔피언별 팀라인별 제어와드,분당피해,분당골드,데스,타워피해,아군보호막,회복량,(ad,ap,ture)뎀 비중 
import pandas as pd
import numpy as np

# 1. 파일 경로 설정 및 데이터 불러오기
file_path = r"D:\바탕화면\데이터 분석\프로젝트\04 최종 프로젝트\구글 드라이브\details\전처리 후\패치 버전 15미만 제외\details_kr_part1_flattened_v15_16.csv"
df = pd.read_csv(file_path)

# 2. 필수 데이터 결측치 제거 및 필터링 (15분 이상, 정상 종료)
df_valid = df.dropna(subset=['championName', 'teamPosition', 'win', 'gameDuration', 'endOfGameResult']).copy()

cond_duration = df_valid['gameDuration'] >= 900
cond_result = df_valid['endOfGameResult'] == 'GameComplete'
df_filtered = df_valid[cond_duration & cond_result].copy()
df_filtered['win'] = df_filtered['win'].astype(int)

# 3. 추가 파생 변수 생성: AD / AP / 고정 데미지 비중(%)
total_dmg = df_filtered['totalDamageDealtToChampions'].replace(0, np.nan)
df_filtered['ad_ratio'] = (df_filtered['physicalDamageDealtToChampions'] / total_dmg) * 100
df_filtered['ap_ratio'] = (df_filtered['magicDamageDealtToChampions'] / total_dmg) * 100
df_filtered['true_ratio'] = (df_filtered['trueDamageDealtToChampions'] / total_dmg) * 100

# 4. [기본 지표 계산] 전체 매치 수, 픽률, 승률
match_count = pd.crosstab(index=df_filtered['championName'], columns=df_filtered['teamPosition'])
pick_rate = pd.crosstab(index=df_filtered['championName'], columns=df_filtered['teamPosition'], normalize='index') * 100
win_rate = pd.pivot_table(data=df_filtered, values='win', index='championName', columns='teamPosition', aggfunc='mean') * 100

# 5. [심화 지표 계산] 출력 순서를 고려하여 딕셔너리 구성 (총 16개 지표)
metric_names = {
    'goldEarned': '골드평균',
    'totalDamageDealtToChampions': '총딜평균',
    'physicalDamageDealtToChampions': 'AD딜평균',
    'ad_ratio': 'AD딜비중(%)',
    'magicDamageDealtToChampions': 'AP딜평균',
    'ap_ratio': 'AP딜비중(%)',
    'trueDamageDealtToChampions': '고정딜평균',
    'true_ratio': '고정딜비중(%)',
    'visionScore': '시야평균',
    'ch_controlWardsPlaced': '제어와드평균',
    'ch_damagePerMinute': 'DPM',
    'ch_goldPerMinute': 'GPM',
    'deaths': '데스평균',
    'damageDealtToBuildings': '건물딜평균',
    'totalDamageShieldedOnTeammates': '아군보호막평균',
    'totalHealsOnTeammates': '아군회복평균'
}

valid_metrics = [m for m in metric_names.keys() if m in df_filtered.columns]
grouped_avg = df_filtered.groupby(['championName', 'teamPosition'])[valid_metrics].mean()
pivoted_avg = grouped_avg.unstack(level='teamPosition')

# 6. 모든 지표를 하나의 데이터프레임으로 조립
desired_order = ['TOP', 'JUNGLE', 'MIDDLE', 'BOTTOM', 'UTILITY']

match_count = match_count.reindex(columns=desired_order).fillna(0)
pick_rate = pick_rate.reindex(columns=desired_order).fillna(0)
win_rate = win_rate.reindex(columns=desired_order).fillna(0)

result_df = pd.DataFrame(index=pick_rate.index)

for pos in desired_order:
    # 6-1. 기본 지표
    result_df[f'{pos}_매치수'] = match_count[pos].astype(int)
    result_df[f'{pos}_픽률(%)'] = pick_rate[pos].round(2)
    result_df[f'{pos}_승률(%)'] = win_rate[pos].round(2)
    
    # 6-2. 16가지 심화 지표 루프 적용 (딕셔너리 순서대로 나란히 생성됨)
    for m_key, m_kor in metric_names.items():
        if m_key in valid_metrics and (m_key, pos) in pivoted_avg.columns:
            result_df[f'{pos}_{m_kor}'] = pivoted_avg[(m_key, pos)].fillna(0).round(2)
        else:
            result_df[f'{pos}_{m_kor}'] = 0.0

# 7. 챔피언별 픽률 1위(주 포지션) 데이터 비우기 (오프메타 픽 분석용)
main_positions = df_filtered.groupby(['championName', 'teamPosition']).size().groupby(level=0).idxmax().apply(lambda x: x[1])

for champ, main_pos in main_positions.items():
    if champ in result_df.index:
        cols_to_clear = [f'{main_pos}_매치수', f'{main_pos}_픽률(%)', f'{main_pos}_승률(%)']
        cols_to_clear += [f'{main_pos}_{m_kor}' for m_kor in metric_names.values()]
        
        # 1위 포지션의 모든 셀을 빈 값(NaN)으로 만듭니다.
        result_df.loc[champ, cols_to_clear] = np.nan

# 8. 결과 확인 및 저장
print("딜 비중 옆에 실제 딜 수치까지 추가 완료! (상위 10개 챔피언 출력):")
display(result_df.head(10))

output_path = r"D:\바탕화면\데이터 분석\프로젝트\04 최종 프로젝트\구글 드라이브\details\전처리 후\패치 버전 15미만 제외\champion_comprehensive_extended_offmeta_kr.csv"
result_df.to_csv(output_path, encoding='utf-8-sig')

print(f"\n최종 통합 데이터가 성공적으로 저장되었습니다:\n{output_path}")

딜 비중 옆에 실제 딜 수치까지 추가 완료! (상위 10개 챔피언 출력):


,TOP_매치수,TOP_픽률(%),TOP_승률(%),TOP_골드평균,TOP_총딜평균,TOP_AD딜평균,TOP_AD딜비중(%),TOP_AP딜평균,TOP_AP딜비중(%),TOP_고정딜평균,...,UTILITY_고정딜평균,UTILITY_고정딜비중(%),UTILITY_시야평균,UTILITY_제어와드평균,UTILITY_DPM,UTILITY_GPM,UTILITY_데스평균,UTILITY_건물딜평균,UTILITY_아군보호막평균,UTILITY_아군회복평균
championName,,,,,,,,,,,,,,,,,,,,,
Aatrox,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,761.91,5.04,44.27,0.73,560.63,317.95,7.73,1543.36,0.00,19.82
Ahri,20.0,1.20,60.00,12563.60,27640.15,2256.20,9.45,19052.95,67.78,6330.00,...,3965.18,24.87,54.68,2.26,572.92,308.21,7.39,1199.47,0.00,50.74
Akali,385.0,17.41,51.69,11702.79,26015.48,1713.11,7.06,22810.05,87.13,1491.39,...,1628.50,8.78,45.00,0.00,762.20,362.77,7.00,611.25,0.00,0.00
Akshan,11.0,4.45,45.45,14114.82,27836.36,22263.73,81.43,3738.73,12.87,1832.91,...,1336.00,7.07,38.17,1.17,670.10,396.56,7.17,1241.50,0.00,0.00
Alistar,20.0,2.30,30.00,9067.05,16714.60,1506.40,10.07,14492.75,85.93,714.75,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Ambessa,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,1887.50,7.45,35.50,1.00,803.40,339.04,7.00,4635.50,0.00,0.00
Amumu,3.0,0.53,33.33,9952.00,17724.33,2076.00,12.25,14309.33,80.26,1339.00,...,1828.63,14.52,58.37,3.77,408.89,291.81,7.35,1013.56,339.66,230.41
Anivia,33.0,7.69,45.45,10932.70,21632.55,1697.67,8.67,18768.91,85.98,1165.03,...,1203.06,7.62,65.24,3.78,524.97,312.29,6.24,1773.59,0.00,119.88
Annie,22.0,6.65,40.91,11184.77,26106.23,1253.73,5.75,23871.86,90.52,979.73,...,1042.86,5.79,68.21,4.14,651.69,321.22,6.88,1528.45,2054.90,71.43



최종 통합 데이터가 성공적으로 저장되었습니다:
D:\바탕화면\데이터 분석\프로젝트\04 최종 프로젝트\구글 드라이브\details\전처리 후\패치 버전 15미만 제외\champion_comprehensive_extended_offmeta_kr.csv


In [3]:
#(role_performance_extended_kr.csv) 팀라인별 승패별 골드,딜량,제어,시야점
import pandas as pd
import numpy as np

# 1. 파일 경로 설정 및 데이터 불러오기
file_path = r"D:\바탕화면\데이터 분석\프로젝트\04 최종 프로젝트\구글 드라이브\details\전처리 후\패치 버전 15미만 제외\details_kr_part1_flattened_v15_16.csv"
df = pd.read_csv(file_path)

# 2. 필수 데이터 결측치 제거 및 필터링 (15분 이상, 정상 종료)
df_valid = df.dropna(subset=['teamPosition', 'win', 'gameDuration', 'endOfGameResult']).copy()

cond_duration = df_valid['gameDuration'] >= 900
cond_result = df_valid['endOfGameResult'] == 'GameComplete'
df_filtered = df_valid[cond_duration & cond_result].copy()
df_filtered['win'] = df_filtered['win'].astype(int)

# 3. 추가 파생 변수 생성: AD / AP / 고정 데미지 비중(%)
total_dmg = df_filtered['totalDamageDealtToChampions'].replace(0, np.nan)
df_filtered['ad_ratio'] = (df_filtered['physicalDamageDealtToChampions'] / total_dmg) * 100
df_filtered['ap_ratio'] = (df_filtered['magicDamageDealtToChampions'] / total_dmg) * 100
df_filtered['true_ratio'] = (df_filtered['trueDamageDealtToChampions'] / total_dmg) * 100

# 4. 분석할 주요 지표 설정 (기존 + 추가된 모든 심화 지표)
metric_mapping = {
    'goldEarned': '골드',
    'totalDamageDealtToChampions': '총딜',
    'ch_teamDamagePercentage': '팀내_딜비중(%)',
    'ch_damagePerMinute': 'DPM',
    'ch_goldPerMinute': 'GPM',
    'deaths': '데스',
    'visionScore': '시야',
    'ch_controlWardsPlaced': '제어와드',
    'damageDealtToBuildings': '건물딜',
    'totalDamageShieldedOnTeammates': '아군보호막',
    'totalHealsOnTeammates': '아군회복',
    'physicalDamageDealtToChampions': 'AD딜',
    'ad_ratio': 'AD비중(%)',
    'magicDamageDealtToChampions': 'AP딜',
    'ap_ratio': 'AP비중(%)',
    'trueDamageDealtToChampions': '고정딜',
    'true_ratio': '고정비중(%)'
}

# 데이터프레임에 존재하는 컬럼만 필터링
valid_metrics = [m for m in metric_mapping.keys() if m in df_filtered.columns]

# 5. 승패 및 포지션별 집계 (평균, 중앙값, 표준편차)
agg_funcs = ['mean', 'median', 'std']
grouped_stats = df_filtered.groupby(['teamPosition', 'win'])[valid_metrics].agg(agg_funcs)

# 멀티인덱스로 묶인 컬럼명을 하나로 평탄화 (예: goldEarned_mean -> 골드_평균)
new_columns = []
for col, func in grouped_stats.columns:
    kor_name = metric_mapping.get(col, col)
    func_kor = {'mean': '평균', 'median': '중앙값', 'std': '표준편차'}.get(func, func)
    new_columns.append(f"{kor_name}_{func_kor}")
    
grouped_stats.columns = new_columns
grouped_stats = grouped_stats.reset_index()

# 6. 보기 좋게 데이터 정제 (한글화 및 순서 정렬)
grouped_stats['win'] = grouped_stats['win'].map({1: '승리', 0: '패배'})

# 포지션 정렬 (탑 -> 정글 -> 미드 -> 원딜 -> 유틸)
desired_order = ['TOP', 'JUNGLE', 'MIDDLE', 'BOTTOM', 'UTILITY']
grouped_stats['teamPosition'] = pd.Categorical(grouped_stats['teamPosition'], categories=desired_order, ordered=True)
grouped_stats = grouped_stats.sort_values(by=['teamPosition', 'win'], ascending=[True, False])

# 보기 좋게 컬럼 이름 변경 (teamPosition -> 포지션, win -> 승패)
grouped_stats.rename(columns={'teamPosition': '포지션', 'win': '승패'}, inplace=True)

# 소수점 둘째 자리까지 반올림
numeric_cols = grouped_stats.select_dtypes(include=['float64', 'int64']).columns
grouped_stats[numeric_cols] = grouped_stats[numeric_cols].round(2)

# 7. 결과 확인
print("조건 적용 완료! (포지션/승패별 심화 지표 분포):")
display(grouped_stats.iloc[:, :15]) # 컬럼이 너무 많아 앞부분 일부만 출력

# 8. 결과를 CSV 파일로 저장
output_path = r"D:\바탕화면\데이터 분석\프로젝트\04 최종 프로젝트\구글 드라이브\details\전처리 후\패치 버전 15미만 제외\role_performance_extended_kr.csv"
grouped_stats.to_csv(output_path, index=False, encoding='utf-8-sig')

print(f"\n심화 지표가 모두 포함된 역할/성과 분포 데이터가 성공적으로 저장되었습니다:\n{output_path}")

조건 적용 완료! (포지션/승패별 심화 지표 분포):


,포지션,승패,골드_평균,골드_중앙값,골드_표준편차,총딜_평균,총딜_중앙값,총딜_표준편차,팀내_딜비중(%)_평균,팀내_딜비중(%)_중앙값,팀내_딜비중(%)_표준편차,DPM_평균,DPM_중앙값,DPM_표준편차,GPM_평균
6,TOP,패배,10749.86,10541.5,3733.26,21691.56,19534.5,11981.89,0.23,0.22,0.07,717.52,684.94,278.62,365.12
7,TOP,승리,12553.81,12468.0,3621.76,25826.54,23981.0,12642.45,0.23,0.22,0.07,869.92,830.03,313.04,433.95
2,JUNGLE,패배,11219.93,10993.0,3652.34,17870.50,15849.5,10804.72,0.18,0.18,0.06,583.73,546.47,258.28,382.92
3,JUNGLE,승리,13093.40,13084.0,3696.23,21593.91,19773.0,11867.02,0.18,0.18,0.06,715.86,676.42,294.23,451.85
4,MIDDLE,패배,10874.82,10699.0,3652.32,21715.08,19783.0,11977.16,0.23,0.22,0.06,714.90,682.33,274.32,370.13
5,MIDDLE,승리,12486.20,12411.0,3523.81,25958.58,24283.0,12583.34,0.23,0.22,0.07,871.02,838.55,302.91,431.51
0,BOTTOM,패배,11769.77,11500.0,4152.69,21669.64,19097.0,13145.25,0.22,0.22,0.07,707.47,662.46,305.69,398.91
1,BOTTOM,승리,13848.60,13722.0,4103.68,27424.08,25047.0,14410.20,0.24,0.23,0.08,916.57,868.36,359.08,477.39
8,UTILITY,패배,8144.07,7942.0,2600.45,12788.86,10446.5,8761.82,0.13,0.12,0.06,422.47,367.57,223.43,278.69
9,UTILITY,승리,9271.13,9185.0,2539.29,14175.40,11774.0,9098.40,0.12,0.11,0.06,476.23,415.51,240.90,320.84



심화 지표가 모두 포함된 역할/성과 분포 데이터가 성공적으로 저장되었습니다:
D:\바탕화면\데이터 분석\프로젝트\04 최종 프로젝트\구글 드라이브\details\전처리 후\패치 버전 15미만 제외\role_performance_extended_kr.csv


### 폴더 순환

In [ ]:
#role_performance_distribution
import os
import glob
import pandas as pd
import numpy as np

# 1. 기본 폴더 경로 설정 (경로를 정확히 맞춰주세요)
base_dir = r"D:\바탕화면\데이터 분석\프로젝트\04 최종 프로젝트\구글 드라이브\details\전처리 후\패치 버전 15미만 제외"

# 2. 폴더 내에서 분석할 원본 파일들을 자동으로 탐색 (패턴 매칭)
search_pattern = os.path.join(base_dir, "details_*_part1_flattened_v15_16.csv")
file_list = glob.glob(search_pattern)

if not file_list:
    print("해당 폴더에 조건에 맞는 파일이 존재하지 않습니다. 경로와 파일명을 확인해주세요.")
else:
    print(f"총 {len(file_list)}개의 파일을 찾아 분석을 시작합니다...\n")

# 3. 분석할 지표 및 한글 매핑 딕셔너리
metric_mapping = {
    'goldEarned': '골드', 'totalDamageDealtToChampions': '총딜',
    'ch_teamDamagePercentage': '팀내_딜비중(%)', 'ch_damagePerMinute': 'DPM',
    'ch_goldPerMinute': 'GPM', 'deaths': '데스', 'visionScore': '시야',
    'ch_controlWardsPlaced': '제어와드', 'damageDealtToBuildings': '건물딜',
    'totalDamageShieldedOnTeammates': '아군보호막', 'totalHealsOnTeammates': '아군회복',
    'physicalDamageDealtToChampions': 'AD딜', 'ad_ratio': 'AD비중(%)',
    'magicDamageDealtToChampions': 'AP딜', 'ap_ratio': 'AP비중(%)',
    'trueDamageDealtToChampions': '고정딜', 'true_ratio': '고정비중(%)'
}
desired_order = ['TOP', 'JUNGLE', 'MIDDLE', 'BOTTOM', 'UTILITY']

# 4. 발견된 모든 파일에 대해 순환(Loop)하며 데이터 처리
for input_path in file_list:
    # 파일명에서 자동으로 서버명 추출 (예: details_kr_part1... -> kr)
    filename = os.path.basename(input_path)
    server = filename.split('_')[1] 
    
    output_path = os.path.join(base_dir, f"role_performance_distribution_{server}.csv")
    print(f"[{server}] 데이터 처리 시작...")
    
    # 데이터 불러오기
    df = pd.read_csv(input_path)
    
    # 필수 결측치 제거 및 필터링 (15분 이상, 정상 종료)
    df_valid = df.dropna(subset=['teamPosition', 'win', 'gameDuration', 'endOfGameResult']).copy()
    cond_duration = df_valid['gameDuration'] >= 900
    cond_result = df_valid['endOfGameResult'] == 'GameComplete'
    df_filtered = df_valid[cond_duration & cond_result].copy()
    df_filtered['win'] = df_filtered['win'].astype(int)
    
    # 만약 필터링 후 데이터가 없다면 다음 파일로 넘어감
    if df_filtered.empty:
        print(f"[{server}] 유효한 데이터가 없어 건너뜁니다.\n")
        continue
    
    # 데미지 비중 파생 변수 생성
    total_dmg = df_filtered['totalDamageDealtToChampions'].replace(0, np.nan)
    df_filtered['ad_ratio'] = (df_filtered['physicalDamageDealtToChampions'] / total_dmg) * 100
    df_filtered['ap_ratio'] = (df_filtered['magicDamageDealtToChampions'] / total_dmg) * 100
    df_filtered['true_ratio'] = (df_filtered['trueDamageDealtToChampions'] / total_dmg) * 100
    
    valid_metrics = [m for m in metric_mapping.keys() if m in df_filtered.columns]
    
    # 집계 (평균, 중앙값, 표준편차)
    grouped_stats = df_filtered.groupby(['teamPosition', 'win'])[valid_metrics].agg(['mean', 'median', 'std'])
    
    # 멀티인덱스 컬럼 평탄화
    new_columns = []
    for col, func in grouped_stats.columns:
        kor_name = metric_mapping.get(col, col)
        func_kor = {'mean': '평균', 'median': '중앙값', 'std': '표준편차'}.get(func, func)
        new_columns.append(f"{kor_name}_{func_kor}")
        
    grouped_stats.columns = new_columns
    grouped_stats = grouped_stats.reset_index()
    
    # 데이터 정렬 및 정제
    grouped_stats['win'] = grouped_stats['win'].map({1: '승리', 0: '패배'})
    grouped_stats['teamPosition'] = pd.Categorical(grouped_stats['teamPosition'], categories=desired_order, ordered=True)
    grouped_stats = grouped_stats.sort_values(by=['teamPosition', 'win'], ascending=[True, False])
    grouped_stats.rename(columns={'teamPosition': '포지션', 'win': '승패'}, inplace=True)
    
    numeric_cols = grouped_stats.select_dtypes(include=['float64', 'int64']).columns
    grouped_stats[numeric_cols] = grouped_stats[numeric_cols].round(2)
    
    # 결과 저장
    grouped_stats.to_csv(output_path, index=False, encoding='utf-8-sig')
    print(f"[{server}] 처리 완료 및 저장: {output_path}\n")

print("✨ 지정된 폴더 안의 모든 파일 처리가 완료되었습니다!")

총 14개의 파일을 찾아 분석을 시작합니다...

[br1] 데이터 처리 시작...
[br1] 처리 완료 및 저장: D:\바탕화면\데이터 분석\프로젝트\04 최종 프로젝트\구글 드라이브\details\전처리 후\패치 버전 15미만 제외\role_performance_distribution_br1.csv

[eun1] 데이터 처리 시작...
[eun1] 처리 완료 및 저장: D:\바탕화면\데이터 분석\프로젝트\04 최종 프로젝트\구글 드라이브\details\전처리 후\패치 버전 15미만 제외\role_performance_distribution_eun1.csv

[euw1] 데이터 처리 시작...
[euw1] 처리 완료 및 저장: D:\바탕화면\데이터 분석\프로젝트\04 최종 프로젝트\구글 드라이브\details\전처리 후\패치 버전 15미만 제외\role_performance_distribution_euw1.csv

[jp1] 데이터 처리 시작...
[jp1] 처리 완료 및 저장: D:\바탕화면\데이터 분석\프로젝트\04 최종 프로젝트\구글 드라이브\details\전처리 후\패치 버전 15미만 제외\role_performance_distribution_jp1.csv

[kr] 데이터 처리 시작...
[kr] 처리 완료 및 저장: D:\바탕화면\데이터 분석\프로젝트\04 최종 프로젝트\구글 드라이브\details\전처리 후\패치 버전 15미만 제외\role_performance_distribution_kr.csv

[la1] 데이터 처리 시작...
[la1] 처리 완료 및 저장: D:\바탕화면\데이터 분석\프로젝트\04 최종 프로젝트\구글 드라이브\details\전처리 후\패치 버전 15미만 제외\role_performance_distribution_la1.csv

[la2] 데이터 처리 시작...
[la2] 처리 완료 및 저장: D:\바탕화면\데이터 분석\프로젝트\04 최종 프로젝트\구글 드라이브\details\전처리 후\패치 버전 15미만 제외\role_pe

In [4]:
#champion_comprehensive_stats
import os
import glob
import pandas as pd
import numpy as np

# 1. 기본 폴더 경로 설정 (경로를 정확히 맞춰주세요)
base_dir = r"D:\바탕화면\데이터 분석\프로젝트\04 최종 프로젝트\구글 드라이브\details\전처리 후\패치 버전 15미만 제외"

# 2. 폴더 내에서 분석할 원본 파일들을 자동으로 탐색
search_pattern = os.path.join(base_dir, "details_*_part1_flattened_v15_16.csv")
file_list = glob.glob(search_pattern)

if not file_list:
    print("해당 폴더에 조건에 맞는 파일이 존재하지 않습니다. 경로와 파일명을 확인해주세요.")
else:
    print(f"총 {len(file_list)}개의 파일을 찾아 챔피언 기본 통합 분석을 시작합니다...\n")

desired_order = ['TOP', 'JUNGLE', 'MIDDLE', 'BOTTOM', 'UTILITY']

# 추출할 3가지 기본 심화 지표
base_metrics_mapping = {
    'goldEarned': '골드평균', 
    'totalDamageDealtToChampions': '딜평균', 
    'visionScore': '시야평균'
}

# 3. 발견된 모든 파일에 대해 순환(Loop) 처리
for input_path in file_list:
    # 파일명에서 자동으로 서버명 추출 (예: details_kr_part1... -> kr)
    filename = os.path.basename(input_path)
    server = filename.split('_')[1] 
    print(f"[{server}] 데이터 처리 시작...")
    
    # 데이터 불러오기
    df = pd.read_csv(input_path)
    
    # 필수 결측치 제거 및 필터링 (15분 이상, 정상 종료)
    df_valid = df.dropna(subset=['championName', 'teamPosition', 'win', 'gameDuration', 'endOfGameResult']).copy()
    cond_duration = df_valid['gameDuration'] >= 900
    cond_result = df_valid['endOfGameResult'] == 'GameComplete'
    df_filtered = df_valid[cond_duration & cond_result].copy()
    
    if df_filtered.empty:
        print(f"[{server}] 유효한 데이터가 없어 건너뜁니다.\n")
        continue
        
    df_filtered['win'] = df_filtered['win'].astype(int)
    
    # --------------------------------------------------
    # [지표 계산]
    # --------------------------------------------------
    # 1) 매치수, 픽률, 승률
    match_count = pd.crosstab(index=df_filtered['championName'], columns=df_filtered['teamPosition']).reindex(columns=desired_order).fillna(0)
    pick_rate = (pd.crosstab(index=df_filtered['championName'], columns=df_filtered['teamPosition'], normalize='index') * 100).reindex(columns=desired_order).fillna(0)
    win_rate = (pd.pivot_table(data=df_filtered, values='win', index='championName', columns='teamPosition', aggfunc='mean') * 100).reindex(columns=desired_order).fillna(0)
    
    # 2) 골드, 딜, 시야
    valid_metrics = [m for m in base_metrics_mapping.keys() if m in df_filtered.columns]
    pivoted_avg = df_filtered.groupby(['championName', 'teamPosition'])[valid_metrics].mean().unstack(level='teamPosition')
    
    # --------------------------------------------------
    # [데이터프레임 조립]
    # --------------------------------------------------
    df_base = pd.DataFrame(index=pick_rate.index)
    
    for pos in desired_order:
        df_base[f'{pos}_매치수'] = match_count[pos].astype(int)
        df_base[f'{pos}_픽률(%)'] = pick_rate[pos].round(2)
        df_base[f'{pos}_승률(%)'] = win_rate[pos].round(2)
        
        for k, v in base_metrics_mapping.items():
            if k in valid_metrics and (k, pos) in pivoted_avg.columns:
                df_base[f'{pos}_{v}'] = pivoted_avg[(k, pos)].fillna(0).round(1)
            else:
                df_base[f'{pos}_{v}'] = 0.0
                
    # 결과 저장
    output_path = os.path.join(base_dir, f"champion_comprehensive_stats_{server}.csv")
    df_base.to_csv(output_path, encoding='utf-8-sig')
    print(f"[{server}] 처리 완료 및 저장: {output_path}\n")

print("✨ 지정된 폴더 안의 모든 파일 처리가 완료되었습니다!")

총 14개의 파일을 찾아 챔피언 기본 통합 분석을 시작합니다...

[br1] 데이터 처리 시작...
[br1] 처리 완료 및 저장: D:\바탕화면\데이터 분석\프로젝트\04 최종 프로젝트\구글 드라이브\details\전처리 후\패치 버전 15미만 제외\champion_comprehensive_stats_br1.csv

[eun1] 데이터 처리 시작...
[eun1] 처리 완료 및 저장: D:\바탕화면\데이터 분석\프로젝트\04 최종 프로젝트\구글 드라이브\details\전처리 후\패치 버전 15미만 제외\champion_comprehensive_stats_eun1.csv

[euw1] 데이터 처리 시작...
[euw1] 처리 완료 및 저장: D:\바탕화면\데이터 분석\프로젝트\04 최종 프로젝트\구글 드라이브\details\전처리 후\패치 버전 15미만 제외\champion_comprehensive_stats_euw1.csv

[jp1] 데이터 처리 시작...
[jp1] 처리 완료 및 저장: D:\바탕화면\데이터 분석\프로젝트\04 최종 프로젝트\구글 드라이브\details\전처리 후\패치 버전 15미만 제외\champion_comprehensive_stats_jp1.csv

[kr] 데이터 처리 시작...
[kr] 처리 완료 및 저장: D:\바탕화면\데이터 분석\프로젝트\04 최종 프로젝트\구글 드라이브\details\전처리 후\패치 버전 15미만 제외\champion_comprehensive_stats_kr.csv

[la1] 데이터 처리 시작...
[la1] 처리 완료 및 저장: D:\바탕화면\데이터 분석\프로젝트\04 최종 프로젝트\구글 드라이브\details\전처리 후\패치 버전 15미만 제외\champion_comprehensive_stats_la1.csv

[la2] 데이터 처리 시작...
[la2] 처리 완료 및 저장: D:\바탕화면\데이터 분석\프로젝트\04 최종 프로젝트\구글 드라이브\details\전처리 후\패치 버전 15미만 제외\cha

In [5]:
#champion_comprehensive_offmeta
import os
import glob
import pandas as pd
import numpy as np

# 1. 기본 폴더 경로 설정 (경로를 정확히 맞춰주세요)
base_dir = r"D:\바탕화면\데이터 분석\프로젝트\04 최종 프로젝트\구글 드라이브\details\전처리 후\패치 버전 15미만 제외"

# 2. 폴더 내에서 분석할 원본 파일들을 자동으로 탐색
search_pattern = os.path.join(base_dir, "details_*_part1_flattened_v15_16.csv")
file_list = glob.glob(search_pattern)

if not file_list:
    print("해당 폴더에 조건에 맞는 파일이 존재하지 않습니다. 경로와 파일명을 확인해주세요.")
else:
    print(f"총 {len(file_list)}개의 파일을 찾아 오프메타 챔피언 기본 분석을 시작합니다...\n")

desired_order = ['TOP', 'JUNGLE', 'MIDDLE', 'BOTTOM', 'UTILITY']

base_metrics_mapping = {
    'goldEarned': '골드평균', 
    'totalDamageDealtToChampions': '딜평균', 
    'visionScore': '시야평균'
}

# 3. 발견된 모든 파일에 대해 순환(Loop) 처리
for input_path in file_list:
    filename = os.path.basename(input_path)
    server = filename.split('_')[1] 
    print(f"[{server}] 데이터 처리 시작...")
    
    # 데이터 불러오기 및 필터링
    df = pd.read_csv(input_path)
    df_valid = df.dropna(subset=['championName', 'teamPosition', 'win', 'gameDuration', 'endOfGameResult']).copy()
    
    cond_duration = df_valid['gameDuration'] >= 900
    cond_result = df_valid['endOfGameResult'] == 'GameComplete'
    df_filtered = df_valid[cond_duration & cond_result].copy()
    
    if df_filtered.empty:
        print(f"[{server}] 유효한 데이터가 없어 건너뜁니다.\n")
        continue
        
    df_filtered['win'] = df_filtered['win'].astype(int)
    
    # --------------------------------------------------
    # [지표 계산]
    # --------------------------------------------------
    match_count = pd.crosstab(index=df_filtered['championName'], columns=df_filtered['teamPosition']).reindex(columns=desired_order).fillna(0)
    pick_rate = (pd.crosstab(index=df_filtered['championName'], columns=df_filtered['teamPosition'], normalize='index') * 100).reindex(columns=desired_order).fillna(0)
    win_rate = (pd.pivot_table(data=df_filtered, values='win', index='championName', columns='teamPosition', aggfunc='mean') * 100).reindex(columns=desired_order).fillna(0)
    
    valid_metrics = [m for m in base_metrics_mapping.keys() if m in df_filtered.columns]
    pivoted_avg = df_filtered.groupby(['championName', 'teamPosition'])[valid_metrics].mean().unstack(level='teamPosition')
    
    # --------------------------------------------------
    # [데이터프레임 조립]
    # --------------------------------------------------
    df_base_offmeta = pd.DataFrame(index=pick_rate.index)
    
    for pos in desired_order:
        df_base_offmeta[f'{pos}_매치수'] = match_count[pos].astype(int)
        df_base_offmeta[f'{pos}_픽률(%)'] = pick_rate[pos].round(2)
        df_base_offmeta[f'{pos}_승률(%)'] = win_rate[pos].round(2)
        
        for k, v in base_metrics_mapping.items():
            if k in valid_metrics and (k, pos) in pivoted_avg.columns:
                df_base_offmeta[f'{pos}_{v}'] = pivoted_avg[(k, pos)].fillna(0).round(1)
            else:
                df_base_offmeta[f'{pos}_{v}'] = 0.0

    # --------------------------------------------------
    # [오프메타 처리] 챔피언별 1위 포지션 찾아서 비우기
    # --------------------------------------------------
    main_positions = df_filtered.groupby(['championName', 'teamPosition']).size().groupby(level=0).idxmax().apply(lambda x: x[1])
    
    for champ, main_pos in main_positions.items():
        if champ in df_base_offmeta.index:
            cols_to_clear = [f'{main_pos}_매치수', f'{main_pos}_픽률(%)', f'{main_pos}_승률(%)', f'{main_pos}_골드평균', f'{main_pos}_딜평균', f'{main_pos}_시야평균']
            df_base_offmeta.loc[champ, cols_to_clear] = np.nan
                
    # 결과 저장
    output_path = os.path.join(base_dir, f"champion_comprehensive_offmeta_{server}.csv")
    df_base_offmeta.to_csv(output_path, encoding='utf-8-sig')
    print(f"[{server}] 처리 완료 및 저장: {output_path}\n")

print("✨ 지정된 폴더 안의 모든 파일 처리가 완료되었습니다!")

총 14개의 파일을 찾아 오프메타 챔피언 기본 분석을 시작합니다...

[br1] 데이터 처리 시작...
[br1] 처리 완료 및 저장: D:\바탕화면\데이터 분석\프로젝트\04 최종 프로젝트\구글 드라이브\details\전처리 후\패치 버전 15미만 제외\champion_comprehensive_offmeta_br1.csv

[eun1] 데이터 처리 시작...
[eun1] 처리 완료 및 저장: D:\바탕화면\데이터 분석\프로젝트\04 최종 프로젝트\구글 드라이브\details\전처리 후\패치 버전 15미만 제외\champion_comprehensive_offmeta_eun1.csv

[euw1] 데이터 처리 시작...
[euw1] 처리 완료 및 저장: D:\바탕화면\데이터 분석\프로젝트\04 최종 프로젝트\구글 드라이브\details\전처리 후\패치 버전 15미만 제외\champion_comprehensive_offmeta_euw1.csv

[jp1] 데이터 처리 시작...
[jp1] 처리 완료 및 저장: D:\바탕화면\데이터 분석\프로젝트\04 최종 프로젝트\구글 드라이브\details\전처리 후\패치 버전 15미만 제외\champion_comprehensive_offmeta_jp1.csv

[kr] 데이터 처리 시작...
[kr] 처리 완료 및 저장: D:\바탕화면\데이터 분석\프로젝트\04 최종 프로젝트\구글 드라이브\details\전처리 후\패치 버전 15미만 제외\champion_comprehensive_offmeta_kr.csv

[la1] 데이터 처리 시작...
[la1] 처리 완료 및 저장: D:\바탕화면\데이터 분석\프로젝트\04 최종 프로젝트\구글 드라이브\details\전처리 후\패치 버전 15미만 제외\champion_comprehensive_offmeta_la1.csv

[la2] 데이터 처리 시작...
[la2] 처리 완료 및 저장: D:\바탕화면\데이터 분석\프로젝트\04 최종 프로젝트\구글 드라이브\details\전처리 후\패치 

In [6]:
#champion_comprehensive_extended_offmeta
import os
import glob
import pandas as pd
import numpy as np

# 1. 기본 폴더 경로 설정 (경로를 정확히 맞춰주세요)
base_dir = r"D:\바탕화면\데이터 분석\프로젝트\04 최종 프로젝트\구글 드라이브\details\전처리 후\패치 버전 15미만 제외"

# 2. 폴더 내에서 분석할 원본 파일들을 자동으로 탐색
search_pattern = os.path.join(base_dir, "details_*_part1_flattened_v15_16.csv")
file_list = glob.glob(search_pattern)

if not file_list:
    print("해당 폴더에 조건에 맞는 파일이 존재하지 않습니다. 경로와 파일명을 확인해주세요.")
else:
    print(f"총 {len(file_list)}개의 파일을 찾아 오프메타 챔피언 확장 분석을 시작합니다...\n")

desired_order = ['TOP', 'JUNGLE', 'MIDDLE', 'BOTTOM', 'UTILITY']

# 추출할 16가지 확장 심화 지표
champ_ext_metrics = {
    'goldEarned': '골드평균', 'totalDamageDealtToChampions': '총딜평균',
    'physicalDamageDealtToChampions': 'AD딜평균', 'ad_ratio': 'AD딜비중(%)',
    'magicDamageDealtToChampions': 'AP딜평균', 'ap_ratio': 'AP딜비중(%)',
    'trueDamageDealtToChampions': '고정딜평균', 'true_ratio': '고정딜비중(%)',
    'visionScore': '시야평균', 'ch_controlWardsPlaced': '제어와드평균',
    'ch_damagePerMinute': 'DPM', 'ch_goldPerMinute': 'GPM',
    'deaths': '데스평균', 'damageDealtToBuildings': '건물딜평균',
    'totalDamageShieldedOnTeammates': '아군보호막평균', 'totalHealsOnTeammates': '아군회복평균'
}

# 3. 발견된 모든 파일에 대해 순환(Loop) 처리
for input_path in file_list:
    filename = os.path.basename(input_path)
    server = filename.split('_')[1] 
    print(f"[{server}] 데이터 처리 시작...")
    
    # 데이터 불러오기 및 필터링
    df = pd.read_csv(input_path)
    df_valid = df.dropna(subset=['championName', 'teamPosition', 'win', 'gameDuration', 'endOfGameResult']).copy()
    
    cond_duration = df_valid['gameDuration'] >= 900
    cond_result = df_valid['endOfGameResult'] == 'GameComplete'
    df_filtered = df_valid[cond_duration & cond_result].copy()
    
    if df_filtered.empty:
        print(f"[{server}] 유효한 데이터가 없어 건너뜁니다.\n")
        continue
        
    df_filtered['win'] = df_filtered['win'].astype(int)
    
    # --------------------------------------------------
    # [파생 변수 생성] 데미지 비중
    # --------------------------------------------------
    total_dmg = df_filtered['totalDamageDealtToChampions'].replace(0, np.nan)
    df_filtered['ad_ratio'] = (df_filtered['physicalDamageDealtToChampions'] / total_dmg) * 100
    df_filtered['ap_ratio'] = (df_filtered['magicDamageDealtToChampions'] / total_dmg) * 100
    df_filtered['true_ratio'] = (df_filtered['trueDamageDealtToChampions'] / total_dmg) * 100
    
    # --------------------------------------------------
    # [지표 계산]
    # --------------------------------------------------
    match_count = pd.crosstab(index=df_filtered['championName'], columns=df_filtered['teamPosition']).reindex(columns=desired_order).fillna(0)
    pick_rate = (pd.crosstab(index=df_filtered['championName'], columns=df_filtered['teamPosition'], normalize='index') * 100).reindex(columns=desired_order).fillna(0)
    win_rate = (pd.pivot_table(data=df_filtered, values='win', index='championName', columns='teamPosition', aggfunc='mean') * 100).reindex(columns=desired_order).fillna(0)
    
    valid_metrics = [m for m in champ_ext_metrics.keys() if m in df_filtered.columns]
    pivoted_avg = df_filtered.groupby(['championName', 'teamPosition'])[valid_metrics].mean().unstack(level='teamPosition')
    
    # --------------------------------------------------
    # [데이터프레임 조립]
    # --------------------------------------------------
    df_ext_offmeta = pd.DataFrame(index=pick_rate.index)
    
    for pos in desired_order:
        df_ext_offmeta[f'{pos}_매치수'] = match_count[pos].astype(int)
        df_ext_offmeta[f'{pos}_픽률(%)'] = pick_rate[pos].round(2)
        df_ext_offmeta[f'{pos}_승률(%)'] = win_rate[pos].round(2)
        
        for k, v in champ_ext_metrics.items():
            if k in valid_metrics and (k, pos) in pivoted_avg.columns:
                df_ext_offmeta[f'{pos}_{v}'] = pivoted_avg[(k, pos)].fillna(0).round(2)
            else:
                df_ext_offmeta[f'{pos}_{v}'] = 0.0

    # --------------------------------------------------
    # [오프메타 처리] 챔피언별 1위 포지션 찾아서 비우기
    # --------------------------------------------------
    main_positions = df_filtered.groupby(['championName', 'teamPosition']).size().groupby(level=0).idxmax().apply(lambda x: x[1])
    
    for champ, main_pos in main_positions.items():
        if champ in df_ext_offmeta.index:
            # 삭제할 열 리스트 구성 (기본 3개 + 확장 16개)
            cols_to_clear = [f'{main_pos}_매치수', f'{main_pos}_픽률(%)', f'{main_pos}_승률(%)']
            cols_to_clear += [f'{main_pos}_{v}' for v in champ_ext_metrics.values()]
            
            # 해당 셀의 데이터를 모두 빈 값(NaN)으로 처리
            df_ext_offmeta.loc[champ, cols_to_clear] = np.nan
                
    # 결과 저장
    output_path = os.path.join(base_dir, f"champion_comprehensive_extended_offmeta_{server}.csv")
    df_ext_offmeta.to_csv(output_path, encoding='utf-8-sig')
    print(f"[{server}] 처리 완료 및 저장: {output_path}\n")

print("✨ 지정된 폴더 안의 모든 파일 처리가 완료되었습니다!")

총 14개의 파일을 찾아 오프메타 챔피언 확장 분석을 시작합니다...

[br1] 데이터 처리 시작...
[br1] 처리 완료 및 저장: D:\바탕화면\데이터 분석\프로젝트\04 최종 프로젝트\구글 드라이브\details\전처리 후\패치 버전 15미만 제외\champion_comprehensive_extended_offmeta_br1.csv

[eun1] 데이터 처리 시작...
[eun1] 처리 완료 및 저장: D:\바탕화면\데이터 분석\프로젝트\04 최종 프로젝트\구글 드라이브\details\전처리 후\패치 버전 15미만 제외\champion_comprehensive_extended_offmeta_eun1.csv

[euw1] 데이터 처리 시작...
[euw1] 처리 완료 및 저장: D:\바탕화면\데이터 분석\프로젝트\04 최종 프로젝트\구글 드라이브\details\전처리 후\패치 버전 15미만 제외\champion_comprehensive_extended_offmeta_euw1.csv

[jp1] 데이터 처리 시작...
[jp1] 처리 완료 및 저장: D:\바탕화면\데이터 분석\프로젝트\04 최종 프로젝트\구글 드라이브\details\전처리 후\패치 버전 15미만 제외\champion_comprehensive_extended_offmeta_jp1.csv

[kr] 데이터 처리 시작...
[kr] 처리 완료 및 저장: D:\바탕화면\데이터 분석\프로젝트\04 최종 프로젝트\구글 드라이브\details\전처리 후\패치 버전 15미만 제외\champion_comprehensive_extended_offmeta_kr.csv

[la1] 데이터 처리 시작...
[la1] 처리 완료 및 저장: D:\바탕화면\데이터 분석\프로젝트\04 최종 프로젝트\구글 드라이브\details\전처리 후\패치 버전 15미만 제외\champion_comprehensive_extended_offmeta_la1.csv

[la2] 데이터 처리 시작...
[la2] 처리 완료 및 저장: D:

In [ ]:
#role_performance_extended
import os
import glob
import pandas as pd
import numpy as np

# 1. 기본 폴더 경로 설정 (경로를 정확히 맞춰주세요)
base_dir = r"D:\바탕화면\데이터 분석\프로젝트\04 최종 프로젝트\구글 드라이브\details\전처리 후\패치 버전 15미만 제외"

# 2. 폴더 내에서 분석할 원본 파일들을 자동으로 탐색
search_pattern = os.path.join(base_dir, "details_*_part1_flattened_v15_16.csv")
file_list = glob.glob(search_pattern)

if not file_list:
    print("해당 폴더에 조건에 맞는 파일이 존재하지 않습니다. 경로와 파일명을 확인해주세요.")
else:
    print(f"총 {len(file_list)}개의 파일을 찾아 역할군 심화 성과 분석을 시작합니다...\n")

desired_order = ['TOP', 'JUNGLE', 'MIDDLE', 'BOTTOM', 'UTILITY']

# 추출할 16가지 심화 지표 및 한글 이름 매핑
role_metrics_mapping = {
    'goldEarned': '골드', 'totalDamageDealtToChampions': '총딜',
    'ch_teamDamagePercentage': '팀내_딜비중(%)', 'ch_damagePerMinute': 'DPM',
    'ch_goldPerMinute': 'GPM', 'deaths': '데스', 'visionScore': '시야',
    'ch_controlWardsPlaced': '제어와드', 'damageDealtToBuildings': '건물딜',
    'totalDamageShieldedOnTeammates': '아군보호막', 'totalHealsOnTeammates': '아군회복',
    'physicalDamageDealtToChampions': 'AD딜', 'ad_ratio': 'AD비중(%)',
    'magicDamageDealtToChampions': 'AP딜', 'ap_ratio': 'AP비중(%)',
    'trueDamageDealtToChampions': '고정딜', 'true_ratio': '고정비중(%)'
}

# 3. 발견된 모든 파일에 대해 순환(Loop) 처리
for input_path in file_list:
    filename = os.path.basename(input_path)
    server = filename.split('_')[1] 
    print(f"[{server}] 데이터 처리 시작...")
    
    # 데이터 불러오기 및 필터링
    df = pd.read_csv(input_path)
    df_valid = df.dropna(subset=['teamPosition', 'win', 'gameDuration', 'endOfGameResult']).copy()
    
    cond_duration = df_valid['gameDuration'] >= 900
    cond_result = df_valid['endOfGameResult'] == 'GameComplete'
    df_filtered = df_valid[cond_duration & cond_result].copy()
    
    if df_filtered.empty:
        print(f"[{server}] 유효한 데이터가 없어 건너뜁니다.\n")
        continue
        
    df_filtered['win'] = df_filtered['win'].astype(int)
    
    # --------------------------------------------------
    # [파생 변수 생성] 데미지 비중
    # --------------------------------------------------
    total_dmg = df_filtered['totalDamageDealtToChampions'].replace(0, np.nan)
    df_filtered['ad_ratio'] = (df_filtered['physicalDamageDealtToChampions'] / total_dmg) * 100
    df_filtered['ap_ratio'] = (df_filtered['magicDamageDealtToChampions'] / total_dmg) * 100
    df_filtered['true_ratio'] = (df_filtered['trueDamageDealtToChampions'] / total_dmg) * 100
    
    # --------------------------------------------------
    # [지표 계산 및 집계] 평균, 중앙값, 표준편차
    # --------------------------------------------------
    valid_metrics = [m for m in role_metrics_mapping.keys() if m in df_filtered.columns]
    grouped_stats = df_filtered.groupby(['teamPosition', 'win'])[valid_metrics].agg(['mean', 'median', 'std'])
    
    # --------------------------------------------------
    # [데이터프레임 조립 및 이름 변경]
    # --------------------------------------------------
    new_cols = []
    for col, func in grouped_stats.columns:
        kor_name = role_metrics_mapping.get(col, col)
        func_kor = {'mean': '평균', 'median': '중앙값', 'std': '표준편차'}.get(func, func)
        new_cols.append(f"{kor_name}_{func_kor}")
        
    grouped_stats.columns = new_cols
    grouped_stats = grouped_stats.reset_index()
    
    # 정렬 및 한글화
    grouped_stats['win'] = grouped_stats['win'].map({1: '승리', 0: '패배'})
    grouped_stats['teamPosition'] = pd.Categorical(grouped_stats['teamPosition'], categories=desired_order, ordered=True)
    grouped_stats = grouped_stats.sort_values(by=['teamPosition', 'win'], ascending=[True, False])
    grouped_stats.rename(columns={'teamPosition': '포지션', 'win': '승패'}, inplace=True)
    
    # 소수점 반올림
    num_cols = grouped_stats.select_dtypes(include=['float64', 'int64']).columns
    grouped_stats[num_cols] = grouped_stats[num_cols].round(2)
    
    # 결과 저장
    output_path = os.path.join(base_dir, f"role_performance_extended_{server}.csv")
    grouped_stats.to_csv(output_path, index=False, encoding='utf-8-sig')
    print(f"[{server}] 처리 완료 및 저장: {output_path}\n")

print("✨ 지정된 폴더 안의 모든 파일 처리가 완료되었습니다!")

In [2]:
# details 전체 파일 순환
import os
import glob
import pandas as pd

# 1. 파일 경로 및 탐색 패턴 설정
input_dir = r"D:\바탕화면\데이터 분석\프로젝트\04 최종 프로젝트\구글 드라이브\details\전처리 후\패치 버전 15미만 제외"
search_pattern = os.path.join(input_dir, "details_*_part1_flattened_v15_16.csv")

# 폴더 내 모든 대상 파일 리스트 가져오기
file_list = glob.glob(search_pattern)

if not file_list:
    print("해당 폴더에 조건에 맞는 파일이 존재하지 않습니다. 경로를 확인해주세요.")
else:
    print(f"총 {len(file_list)}개의 파일을 찾아 병합을 시작합니다...\n")

    # 불러온 데이터프레임들을 담을 빈 리스트
    df_list = []
    total_rows = 0

    # 2. 폴더 내 파일 순환하며 데이터 불러오기
    for file_path in file_list:
        # 파일명에서 서버 이름 추출 (예: details_na1_part1_... -> na1 -> NA1)
        filename = os.path.basename(file_path)
        server_name = filename.split('_')[1].upper() 
        
        print(f"[{server_name}] 데이터를 불러오는 중... ({filename})")
        
        # CSV 파일 읽기
        df_temp = pd.read_csv(file_path)
        
        # 출처 서버를 구분할 수 있도록 'server' 컬럼 추가
        df_temp['server'] = server_name
        
        # 리스트에 추가
        df_list.append(df_temp)
        
        # 행 수 기록
        rows = len(df_temp)
        total_rows += rows
        print(f"  -> {rows:,} 행 추가 완료")

    # 3. 데이터프레임 유니온 (리스트에 모인 모든 데이터를 위아래로 한 번에 병합)
    print("\n데이터를 하나로 병합하는 중입니다. (잠시만 기다려주세요...)")
    df_union = pd.concat(df_list, ignore_index=True)

    # 4. 병합 결과 확인
    print(f"\n✅ 병합된 총 데이터 행 수: {len(df_union):,} 개")

    # 5. 병합된 데이터를 새로운 CSV 파일로 저장 (요청하신 경로 그대로 유지)
    output_path = r"D:\바탕화면\데이터 분석\프로젝트\04 최종 프로젝트\구글 드라이브\details\전처리 후\details_union2.csv"
    df_union.to_csv(output_path, index=False, encoding='utf-8-sig')

    print(f"\n유니온(Union)이 완료된 파일이 성공적으로 저장되었습니다:\n{output_path}")

총 14개의 파일을 찾아 병합을 시작합니다...

[BR1] 데이터를 불러오는 중... (details_br1_part1_flattened_v15_16.csv)


C:\Users\Public\Documents\ESTsoft\CreatorTemp\ipykernel_13976\2737702590.py:34: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df_temp['server'] = server_name


  -> 63,930 행 추가 완료
[EUN1] 데이터를 불러오는 중... (details_eun1_part1_flattened_v15_16.csv)


C:\Users\Public\Documents\ESTsoft\CreatorTemp\ipykernel_13976\2737702590.py:34: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df_temp['server'] = server_name


  -> 76,840 행 추가 완료
[EUW1] 데이터를 불러오는 중... (details_euw1_part1_flattened_v15_16.csv)


...

  -> 40,050 행 추가 완료
[TW2] 데이터를 불러오는 중... (details_tw2_part1_flattened_v15_16.csv)


C:\Users\Public\Documents\ESTsoft\CreatorTemp\ipykernel_13976\2737702590.py:34: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df_temp['server'] = server_name


  -> 10,880 행 추가 완료
[VN2] 데이터를 불러오는 중... (details_vn2_part1_flattened_v15_16.csv)


C:\Users\Public\Documents\ESTsoft\CreatorTemp\ipykernel_13976\2737702590.py:34: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df_temp['server'] = server_name


  -> 56,950 행 추가 완료

데이터를 하나로 병합하는 중입니다. (잠시만 기다려주세요...)

✅ 병합된 총 데이터 행 수: 708,960 개

유니온(Union)이 완료된 파일이 성공적으로 저장되었습니다:
D:\바탕화면\데이터 분석\프로젝트\04 최종 프로젝트\구글 드라이브\details\전처리 후\details_union2.csv


In [ ]:
#role_performance_distribution
import os
import glob
import pandas as pd

# 1. 파일 경로 및 탐색 패턴 설정
input_dir = r"D:\바탕화면\데이터 분석\프로젝트\04 최종 프로젝트\구글 드라이브\details\전처리 후\role_performance_distribution"
search_pattern = os.path.join(input_dir, "role_performance_distribution_*.csv")

# 폴더 내 모든 대상 파일 리스트 가져오기
file_list = glob.glob(search_pattern)

if not file_list:
    print("해당 폴더에 조건에 맞는 파일이 존재하지 않습니다. 경로를 확인해주세요.")
else:
    print(f"총 {len(file_list)}개의 파일을 찾아 병합을 시작합니다...\n")

    # 불러온 데이터프레임들을 담을 빈 리스트
    df_list = []

    # 2. 폴더 내 파일 순환하며 데이터 불러오기
    for file_path in file_list:
        # 파일명 가져오기 (예: role_performance_distribution_kr.csv)
        filename = os.path.basename(file_path)
        
        # [수정 부분] 확장자(.csv)를 제거한 순수 파일명 생성
        name_without_ext = os.path.splitext(filename)[0]
        
        # '_'로 스플릿하여 서버 이름 부분만 추출 (4번째 요소)
        # 예: role(0)_performance(1)_distribution(2)_kr(3)
        server_name = name_without_ext.split('_')[3].upper() 
        
        print(f"[{server_name}] 데이터를 불러오는 중... ({filename})")
        
        # CSV 파일 읽기
        df_temp = pd.read_csv(file_path)
        
        # 출처 서버를 구분할 수 있도록 'server' 컬럼 추가 (확장자 없는 깔끔한 이름)
        df_temp['server'] = server_name
        
        # 리스트에 추가
        df_list.append(df_temp)
        
        print(f"  -> {len(df_temp):,} 행 추가 완료")

    # 3. 데이터프레임 유니온
    print("\n데이터를 하나로 병합하는 중입니다...")
    df_union = pd.concat(df_list, ignore_index=True)

    # 4. 병합 결과 확인
    print(f"\n✅ 병합된 총 데이터 행 수: {len(df_union):,} 개")

    # 5. 저장
    output_path = r"D:\바탕화면\데이터 분석\프로젝트\04 최종 프로젝트\구글 드라이브\details\전처리 후\role_performance_distribution_union.csv"
    df_union.to_csv(output_path, index=False, encoding='utf-8-sig')

    print(f"\n유니온(Union)이 완료된 파일이 성공적으로 저장되었습니다:\n{output_path}")

총 14개의 파일을 찾아 병합을 시작합니다...

[BR1] 데이터를 불러오는 중... (role_performance_distribution_br1.csv)
  -> 10 행 추가 완료
[EUN1] 데이터를 불러오는 중... (role_performance_distribution_eun1.csv)
  -> 10 행 추가 완료
[EUW1] 데이터를 불러오는 중... (role_performance_distribution_euw1.csv)
  -> 10 행 추가 완료
[JP1] 데이터를 불러오는 중... (role_performance_distribution_jp1.csv)
  -> 10 행 추가 완료
[KR] 데이터를 불러오는 중... (role_performance_distribution_kr.csv)
  -> 10 행 추가 완료
[LA1] 데이터를 불러오는 중... (role_performance_distribution_la1.csv)
  -> 10 행 추가 완료
[LA2] 데이터를 불러오는 중... (role_performance_distribution_la2.csv)
  -> 10 행 추가 완료
[ME1] 데이터를 불러오는 중... (role_performance_distribution_me1.csv)
  -> 10 행 추가 완료
[NA1] 데이터를 불러오는 중... (role_performance_distribution_na1.csv)
  -> 10 행 추가 완료
[OC1] 데이터를 불러오는 중... (role_performance_distribution_oc1.csv)
  -> 10 행 추가 완료
[RU] 데이터를 불러오는 중... (role_performance_distribution_ru.csv)
  -> 10 행 추가 완료
[TR1] 데이터를 불러오는 중... (role_performance_distribution_tr1.csv)
  -> 10 행 추가 완료
[TW2] 데이터를 불러오는 중... (role_performance_distribut

In [ ]:
#champion_comprehensive_stats
import os
import glob
import pandas as pd

# 1. 파일 경로 및 탐색 패턴 설정
input_dir = r"D:\바탕화면\데이터 분석\프로젝트\04 최종 프로젝트\구글 드라이브\details\전처리 후\champion_comprehensive_stats"
search_pattern = os.path.join(input_dir, "champion_comprehensive_stats_*.csv")

# 폴더 내 모든 대상 파일 리스트 가져오기
file_list = glob.glob(search_pattern)

if not file_list:
    print("해당 폴더에 조건에 맞는 파일이 존재하지 않습니다. 경로를 확인해주세요.")
else:
    print(f"총 {len(file_list)}개의 파일을 찾아 병합을 시작합니다...\n")

    # 불러온 데이터프레임들을 담을 빈 리스트
    df_list = []

    # 2. 폴더 내 파일 순환하며 데이터 불러오기
    for file_path in file_list:
        # 파일명 가져오기 (예: role_performance_distribution_kr.csv)
        filename = os.path.basename(file_path)
        
        # [수정 부분] 확장자(.csv)를 제거한 순수 파일명 생성
        name_without_ext = os.path.splitext(filename)[0]
        
        # '_'로 스플릿하여 서버 이름 부분만 추출 (4번째 요소)
        # 예: role(0)_performance(1)_distribution(2)_kr(3)
        server_name = name_without_ext.split('_')[3].upper() 
        
        print(f"[{server_name}] 데이터를 불러오는 중... ({filename})")
        
        # CSV 파일 읽기
        df_temp = pd.read_csv(file_path)
        
        # 출처 서버를 구분할 수 있도록 'server' 컬럼 추가 (확장자 없는 깔끔한 이름)
        df_temp['server'] = server_name
        
        # 리스트에 추가
        df_list.append(df_temp)
        
        print(f"  -> {len(df_temp):,} 행 추가 완료")

    # 3. 데이터프레임 유니온
    print("\n데이터를 하나로 병합하는 중입니다...")
    df_union = pd.concat(df_list, ignore_index=True)

    # 4. 병합 결과 확인
    print(f"\n✅ 병합된 총 데이터 행 수: {len(df_union):,} 개")

    # 5. 저장
    output_path = r"D:\바탕화면\데이터 분석\프로젝트\04 최종 프로젝트\구글 드라이브\details\전처리 후\champion_comprehensive_stats_union.csv"
    df_union.to_csv(output_path, index=False, encoding='utf-8-sig')

    print(f"\n유니온(Union)이 완료된 파일이 성공적으로 저장되었습니다:\n{output_path}")

총 14개의 파일을 찾아 병합을 시작합니다...

[BR1] 데이터를 불러오는 중... (champion_comprehensive_stats_br1.csv)
  -> 172 행 추가 완료
[EUN1] 데이터를 불러오는 중... (champion_comprehensive_stats_eun1.csv)
  -> 172 행 추가 완료
[EUW1] 데이터를 불러오는 중... (champion_comprehensive_stats_euw1.csv)
  -> 172 행 추가 완료
[JP1] 데이터를 불러오는 중... (champion_comprehensive_stats_jp1.csv)
  -> 172 행 추가 완료
[KR] 데이터를 불러오는 중... (champion_comprehensive_stats_kr.csv)
  -> 172 행 추가 완료
[LA1] 데이터를 불러오는 중... (champion_comprehensive_stats_la1.csv)
  -> 172 행 추가 완료
[LA2] 데이터를 불러오는 중... (champion_comprehensive_stats_la2.csv)
  -> 172 행 추가 완료
[ME1] 데이터를 불러오는 중... (champion_comprehensive_stats_me1.csv)
  -> 170 행 추가 완료
[NA1] 데이터를 불러오는 중... (champion_comprehensive_stats_na1.csv)
  -> 172 행 추가 완료
[OC1] 데이터를 불러오는 중... (champion_comprehensive_stats_oc1.csv)
  -> 172 행 추가 완료
[RU] 데이터를 불러오는 중... (champion_comprehensive_stats_ru.csv)
  -> 172 행 추가 완료
[TR1] 데이터를 불러오는 중... (champion_comprehensive_stats_tr1.csv)
  -> 172 행 추가 완료
[TW2] 데이터를 불러오는 중... (champion_comprehensive_sta

In [ ]:
#champion_comprehensive_extended_offmeta
import os
import glob
import pandas as pd

# 1. 파일 경로 및 탐색 패턴 설정
input_dir = r"D:\바탕화면\데이터 분석\프로젝트\04 최종 프로젝트\구글 드라이브\details\전처리 후\champion_comprehensive_extended_offmeta"
search_pattern = os.path.join(input_dir, "champion_comprehensive_extended_offmeta_*.csv")

# 폴더 내 모든 대상 파일 리스트 가져오기
file_list = glob.glob(search_pattern)

if not file_list:
    print("해당 폴더에 조건에 맞는 파일이 존재하지 않습니다. 경로를 확인해주세요.")
else:
    print(f"총 {len(file_list)}개의 파일을 찾아 병합을 시작합니다...\n")

    # 불러온 데이터프레임들을 담을 빈 리스트
    df_list = []

    # 2. 폴더 내 파일 순환하며 데이터 불러오기
    for file_path in file_list:
        # 파일명 가져오기 (예: role_performance_distribution_kr.csv)
        filename = os.path.basename(file_path)
        
        # [수정 부분] 확장자(.csv)를 제거한 순수 파일명 생성
        name_without_ext = os.path.splitext(filename)[0]
        
        # '_'로 스플릿하여 서버 이름 부분만 추출 (4번째 요소)
        # 예: role(0)_performance(1)_distribution(2)_kr(3)
        server_name = name_without_ext.split('_')[3].upper() 
        
        print(f"[{server_name}] 데이터를 불러오는 중... ({filename})")
        
        # CSV 파일 읽기
        df_temp = pd.read_csv(file_path)
        
        # 출처 서버를 구분할 수 있도록 'server' 컬럼 추가 (확장자 없는 깔끔한 이름)
        df_temp['server'] = server_name
        
        # 리스트에 추가
        df_list.append(df_temp)
        
        print(f"  -> {len(df_temp):,} 행 추가 완료")

    # 3. 데이터프레임 유니온
    print("\n데이터를 하나로 병합하는 중입니다...")
    df_union = pd.concat(df_list, ignore_index=True)

    # 4. 병합 결과 확인
    print(f"\n✅ 병합된 총 데이터 행 수: {len(df_union):,} 개")

    # 5. 저장
    output_path = r"D:\바탕화면\데이터 분석\프로젝트\04 최종 프로젝트\구글 드라이브\details\전처리 후\champion_comprehensive_extended_offmeta_union.csv"
    df_union.to_csv(output_path, index=False, encoding='utf-8-sig')

    print(f"\n유니온(Union)이 완료된 파일이 성공적으로 저장되었습니다:\n{output_path}")

총 14개의 파일을 찾아 병합을 시작합니다...

[OFFMETA] 데이터를 불러오는 중... (champion_comprehensive_extended_offmeta_br1.csv)
  -> 172 행 추가 완료
[OFFMETA] 데이터를 불러오는 중... (champion_comprehensive_extended_offmeta_eun1.csv)
  -> 172 행 추가 완료
[OFFMETA] 데이터를 불러오는 중... (champion_comprehensive_extended_offmeta_euw1.csv)
  -> 172 행 추가 완료
[OFFMETA] 데이터를 불러오는 중... (champion_comprehensive_extended_offmeta_jp1.csv)
  -> 172 행 추가 완료
[OFFMETA] 데이터를 불러오는 중... (champion_comprehensive_extended_offmeta_kr.csv)
  -> 172 행 추가 완료
[OFFMETA] 데이터를 불러오는 중... (champion_comprehensive_extended_offmeta_la1.csv)
  -> 172 행 추가 완료
[OFFMETA] 데이터를 불러오는 중... (champion_comprehensive_extended_offmeta_la2.csv)
  -> 172 행 추가 완료
[OFFMETA] 데이터를 불러오는 중... (champion_comprehensive_extended_offmeta_me1.csv)
  -> 170 행 추가 완료
[OFFMETA] 데이터를 불러오는 중... (champion_comprehensive_extended_offmeta_na1.csv)
  -> 172 행 추가 완료
[OFFMETA] 데이터를 불러오는 중... (champion_comprehensive_extended_offmeta_oc1.csv)
  -> 172 행 추가 완료
[OFFMETA] 데이터를 불러오는 중... (champion_comprehensive_ex